# Análise Exploratória de Dados: Desempenho Acadêmico de Estudantes

**Disciplina:** Análise Exploratória de Dados (CCA_26_2)
**Grupo:** Arthur da Fonte, Matheus Freire, Matheus Fialho, Pablo Coelho, Rodrigo Vinhas, Bruno Holanda, Vitor Gadelha, Raul Maia

**Problema:** Identificar quais fatores estão mais associados ao desempenho e à aprovação dos alunos, para permitir intervenções antes da prova em vez de só constatar a reprovação depois.

**Fonte dos dados:** [Student Exam Performance & Success Dataset (Kaggle)](https://www.kaggle.com/datasets/mobeenfatimah/student-exam-performance-and-success-dataset)

**Perguntas de investigação:**
1. Quem estuda mais horas tira nota maior? (com retorno decrescente a partir de certo ponto?)
2. A frequência às aulas separa aprovados de reprovados?
3. Dormir pouco derruba o desempenho mesmo em quem estuda muito?

*(Preencher/ajustar conforme necessário — este é um template inicial.)*


## 1. Setup e Carregamento dos Dados

In [ ]:
# Bibliotecas usadas na análise
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

sns.set_style("whitegrid")


In [ ]:
# TODO: ajustar o caminho conforme onde o CSV estiver
# (upload direto no Colab com files.upload(), Google Drive montado, ou pasta local)
CAMINHO_CSV = "student_exam_performance.csv"

df_completo = pd.read_csv(CAMINHO_CSV)
print("Shape original:", df_completo.shape)
df_completo.head()


O dataset original tem 44 colunas. Para esta análise, vamos manter apenas as 5 colunas
mais relevantes para as perguntas de investigação do grupo:

- `exam_score`: nota final da prova (variável de desempenho)
- `sleep_hours`: horas de sono
- `study_hours_per_day`: horas de estudo por dia
- `attendance_percentage`: percentual de frequência às aulas
- `pass_status`: status de aprovação (Pass / Fail)


In [ ]:
colunas_selecionadas = [
    "exam_score",
    "sleep_hours",
    "study_hours_per_day",
    "attendance_percentage",
    "pass_status",
]

df = df_completo[colunas_selecionadas].copy()
df.info()


## 2. Pré-processamento de Dados

### 2.1 Valores nulos/ausentes

In [ ]:
# Verificando quantos valores faltam em cada coluna
nulos = df.isnull().sum()
percentual_nulos = (nulos / len(df) * 100).round(2)
pd.DataFrame({"nulos": nulos, "percentual (%)": percentual_nulos})


**Critério adotado:** se a coluna tiver menos de 5% de valores ausentes, apagamos as linhas
(com 100 mil registros, perder algumas centenas não atrapalha). Se ultrapassar 5%, imputamos
pela mediana (colunas numéricas) ou pela moda / categoria "Não informado" (colunas categóricas).

`attendance_percentage` tem ~9,86% de nulos — acima do limite de 5% — então será imputada pela
mediana, e não descartada.


In [ ]:
mediana_attendance = df["attendance_percentage"].median()
df["attendance_percentage"] = df["attendance_percentage"].fillna(mediana_attendance)

print(f"Mediana usada para imputação: {mediana_attendance}")
print("Nulos restantes no dataset:", df.isnull().sum().sum())


### 2.2 Dados duplicados

In [ ]:
duplicadas = df.duplicated().sum()
print("Linhas duplicadas encontradas:", duplicadas)

# Caso existam, remover:
# df = df.drop_duplicates()


### 2.3 Conversão de tipos

`pass_status` vem como texto ("Pass"/"Fail"). Para poder usá-la em correlação e em métricas
estatísticas numéricas, criamos uma versão binária (`aprovado`: 1 = Pass, 0 = Fail), mantendo a
coluna original para leitura/gráficos.


In [ ]:
df["aprovado"] = (df["pass_status"] == "Pass").astype(int)
df.dtypes


## 3. Análise Estatística

### 3.1 Medidas de Centralização (Média, Mediana, Moda)

In [ ]:
variaveis_numericas = ["exam_score", "sleep_hours", "study_hours_per_day", "attendance_percentage"]

centralizacao = pd.DataFrame({
    "media": df[variaveis_numericas].mean(),
    "mediana": df[variaveis_numericas].median(),
    "moda": df[variaveis_numericas].mode().iloc[0],
})
centralizacao


*(Comentar aqui o que os números indicam: ex. se média e mediana são próximas, distribuição tende a ser simétrica.)*

### 3.2 Medidas de Posição (Quartis)

In [ ]:
quartis = df[variaveis_numericas].quantile([0.25, 0.5, 0.75])
quartis


### 3.3 Medidas de Dispersão (Variância, Desvio Padrão, Amplitude)

In [ ]:
dispersao = pd.DataFrame({
    "variancia": df[variaveis_numericas].var(),
    "desvio_padrao": df[variaveis_numericas].std(),
    "amplitude": df[variaveis_numericas].max() - df[variaveis_numericas].min(),
})
dispersao


### 3.4 Correlação (Pearson)

In [ ]:
colunas_corr = variaveis_numericas + ["aprovado"]
matriz_correlacao = df[colunas_corr].corr(method="pearson")
matriz_correlacao


In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(matriz_correlacao, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlação entre as variáveis")
plt.tight_layout()
plt.show()


### 3.5 Normalização / Padronização

As variáveis estão em escalas bem diferentes (sono vai de ~3 a 11 horas, estudo de 0 a ~11h/dia,
frequência é percentual de 0 a 100). Antes de comparar a magnitude das variáveis entre si — ou de
usá-las em conjunto em qualquer análise sensível a escala — é preciso padronizar, senão a variável
com maior amplitude (`attendance_percentage`) domina artificialmente a comparação.


In [ ]:
scaler = StandardScaler()
df_normalizado = pd.DataFrame(
    scaler.fit_transform(df[variaveis_numericas]),
    columns=[c + "_norm" for c in variaveis_numericas],
)
df_normalizado.describe()


## 4. Respondendo às Perguntas de Investigação

### Pergunta 1 — Quem estuda mais tira nota maior? Existe retorno decrescente?

In [ ]:
df["faixa_estudo"] = pd.cut(
    df["study_hours_per_day"],
    bins=[0, 1, 2, 3, 4, df["study_hours_per_day"].max()],
    labels=["0-1h", "1-2h", "2-3h", "3-4h", "4h+"],
)

media_por_faixa_estudo = df.groupby("faixa_estudo", observed=True)["exam_score"].mean()
media_por_faixa_estudo


In [ ]:
plt.figure(figsize=(7, 4))
sns.barplot(x=media_por_faixa_estudo.index, y=media_por_faixa_estudo.values)
plt.xlabel("Faixa de horas de estudo/dia")
plt.ylabel("Nota média")
plt.title("Nota média por faixa de horas de estudo")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(data=df.sample(3000, random_state=42), x="study_hours_per_day", y="exam_score", alpha=0.3)
plt.xlabel("Horas de estudo/dia")
plt.ylabel("Nota final")
plt.title("Horas de estudo x Nota (amostra de 3000 pontos)")
plt.tight_layout()
plt.show()


*(Escrever aqui a conclusão parcial: a correlação de Pearson calculada na seção 3.4 confirma a direção; observar nas faixas acima se o ganho de nota entre faixas vai diminuindo — sinal de retorno decrescente.)*

### Pergunta 2 — A frequência separa aprovados de reprovados?

In [ ]:
comparacao_aprovacao = df.groupby("pass_status")[variaveis_numericas].mean()
comparacao_aprovacao


In [ ]:
df["faixa_frequencia"] = pd.cut(
    df["attendance_percentage"],
    bins=[0, 60, 75, 90, 100],
    labels=["<60%", "60-75%", "75-90%", "90-100%"],
)

taxa_aprovacao_por_faixa = df.groupby("faixa_frequencia", observed=True)["aprovado"].mean()

plt.figure(figsize=(7, 4))
sns.barplot(x=taxa_aprovacao_por_faixa.index, y=taxa_aprovacao_por_faixa.values)
plt.xlabel("Faixa de frequência")
plt.ylabel("Taxa de aprovação")
plt.title("Taxa de aprovação por faixa de frequência")
plt.tight_layout()
plt.show()


*(Escrever aqui: comparar a diferença de frequência média entre aprovados e reprovados, e comparar o peso da frequência com o de horas de estudo na correlação com `aprovado`.)*

### Pergunta 3 — Dormir pouco prejudica mesmo quem estuda muito?

In [ ]:
# Isolando o grupo que mais estuda (top 25%)
limite_estudo = df["study_hours_per_day"].quantile(0.75)
grupo_estuda_muito = df[df["study_hours_per_day"] >= limite_estudo]

mediana_sono_grupo = grupo_estuda_muito["sleep_hours"].median()
dorme_pouco = grupo_estuda_muito[grupo_estuda_muito["sleep_hours"] < mediana_sono_grupo]
dorme_bem = grupo_estuda_muito[grupo_estuda_muito["sleep_hours"] >= mediana_sono_grupo]

print("Nota média (estuda muito + dorme pouco):", dorme_pouco["exam_score"].mean())
print("Nota média (estuda muito + dorme bem):", dorme_bem["exam_score"].mean())


*(Escrever aqui a conclusão parcial: mesmo dentro de quem mais estuda, há diferença de nota entre quem dorme pouco e quem dorme bem?)*

## 5. Visualizações Gerais

In [ ]:
fig, eixos = plt.subplots(2, 2, figsize=(11, 8))

sns.histplot(df["exam_score"], kde=True, ax=eixos[0, 0])
eixos[0, 0].set_title("Distribuição da nota final")

sns.histplot(df["sleep_hours"], kde=True, ax=eixos[0, 1])
eixos[0, 1].set_title("Distribuição de horas de sono")

sns.histplot(df["study_hours_per_day"], kde=True, ax=eixos[1, 0])
eixos[1, 0].set_title("Distribuição de horas de estudo")

sns.histplot(df["attendance_percentage"], kde=True, ax=eixos[1, 1])
eixos[1, 1].set_title("Distribuição de frequência")

plt.tight_layout()
plt.show()


## 6. Conclusões

*(Cada integrante deve revisar e escrever a conclusão final com as próprias palavras, retomando
as 3 hipóteses da planilha e dizendo se foram confirmadas, parcialmente confirmadas ou refutadas
com base nos números calculados acima.)*

- **Hipótese 1:**
- **Hipótese 2:**
- **Hipótese 3:**

### Limitações
Dataset genérico, sem contexto de país/escola/disciplina — mostra associação, não causa.
